# Import Required Libraries
Import `pandas`, `json`, and `os` modules for file handling and data manipulation.

In [2]:
import os
import json
import csv

print('Current directory:', os.getcwd())
print('Files present:', [f for f in os.listdir('.') if f.endswith('.json')])

Current directory: /workspaces/Bulk_RequestAccess_CollectionLevel
Files present: ['export2.json', 'export1.json']


# Load JSON Files
Read JSON files (`export1.json`, `export2.json`) and parse their contents.

In [5]:
json_files = ['export1.json', 'export2.json']
parsed_json = {}
for json_file in json_files:
    with open(json_file, 'r', encoding='utf-8') as f:
        parsed_json[json_file] = json.load(f)
        print(f"Loaded {json_file}: type={type(parsed_json[json_file]).__name__}")
        if isinstance(parsed_json[json_file], dict):
            keys = list(parsed_json[json_file].keys())
            print('all keys:', keys)
            print('files present:', 'files' in parsed_json[json_file])
            print('datasetVersion present:', 'datasetVersion' in parsed_json[json_file])
            if 'datasetVersion' in parsed_json[json_file]:
                dv = parsed_json[json_file]['datasetVersion']
                print('datasetVersion keys:', list(dv.keys()) if isinstance(dv, dict) else type(dv))
            print('files count:', len(parsed_json[json_file].get('files', [])))
        print()

Loaded export1.json: type=dict
all keys: ['id', 'identifier', 'persistentUrl', 'protocol', 'authority', 'separator', 'publisher', 'publicationDate', 'storageIdentifier', 'datasetType', 'datasetVersion']
files present: False
datasetVersion present: True
datasetVersion keys: ['id', 'datasetId', 'datasetPersistentId', 'datasetType', 'storageIdentifier', 'versionNumber', 'internalVersionNumber', 'versionMinorNumber', 'versionState', 'latestVersionPublishingState', 'UNF', 'lastUpdateTime', 'releaseTime', 'createTime', 'publicationDate', 'citationDate', 'termsOfUse', 'termsOfAccess', 'fileAccessRequest', 'metadataBlocks', 'files', 'citation']
files count: 0

Loaded export2.json: type=dict
all keys: ['id', 'identifier', 'persistentUrl', 'protocol', 'authority', 'separator', 'publisher', 'publicationDate', 'storageIdentifier', 'datasetType', 'datasetVersion']
files present: False
datasetVersion present: True
datasetVersion keys: ['id', 'datasetId', 'datasetPersistentId', 'datasetType', 'storag

# Parse and Flatten JSON Data
Extract and flatten nested JSON structures to prepare data for tabular format.

In [4]:
def extract_title(version):
    citation = version.get('metadataBlocks', {}).get('citation', {})
    fields = citation.get('fields', [])
    for field in fields:
        if field.get('typeName') == 'title':
            return field.get('value')
    return None

rows = []
for json_file, data in parsed_json.items():
    if not isinstance(data, dict):
        continue
    doi = data.get('persistentUrl') or data.get('identifier', '')
    version = data.get('datasetVersion', {})
    title = extract_title(version) or data.get('title', '')
    for file_record in data.get('files', []):
        file_item = file_record.get('dataFile', {})
        rows.append({
            'json_file': json_file,
            'doi': doi,
            'dataset_title': title,
            'file_id': file_item.get('id'),
            'file_name': file_item.get('filename'),
            'restricted': file_record.get('restricted'),
            'file_access_request': file_item.get('fileAccessRequest')
        })

print('Total rows parsed:', len(rows))
rows[:3]

Total rows parsed: 0


[]

# Handle Multiple JSON Files
Iterate through multiple JSON files and combine their data into a single collection.

In [39]:
import subprocess
result = subprocess.run(['python3', 'json_to_csv.py', 'export1.json', 'export2.json', '-o', 'dataset_files.csv'], capture_output=True, text=True)
print('returncode:', result.returncode)
print(result.stdout)
if result.stderr:
    print('stderr:', result.stderr)

returncode: 0
Wrote 6 rows to dataset_files.csv



# Convert to DataFrame
Convert the parsed JSON data into a pandas DataFrame for easier manipulation.

In [ ]:
output_csv = 'dataset_files.csv'
with open(output_csv, 'w', newline='', encoding='utf-8') as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=['json_file','doi','dataset_title','file_id','file_name','restricted','file_access_request'])
    writer.writeheader()
    writer.writerows(rows)
print(f'Wrote CSV to {output_csv}')

# Verify Output
Check the resulting CSV file to confirm data was written correctly and inspect row counts.

In [8]:
output_csv = 'dataset_files.csv'
print('CSV exists:', os.path.exists(output_csv))
print('CSV preview:')
with open(output_csv, 'r', encoding='utf-8') as csvfile:
    for _ in range(5):
        print(csvfile.readline().rstrip())
print('CSV row count:', sum(1 for _ in open(output_csv, 'r', encoding='utf-8')) - 1)

CSV exists: True
CSV preview:
doi,dataset_title,file_id,file_name,restricted,file_access_request
https://doi.org/10.80240/FK2/DOAFHQ,Restricted 2,70218,Demotestmetadata.json,False,False
https://doi.org/10.80240/FK2/DOAFHQ,Restricted 2,70222,ExcelExample_Corr_Reg (1).tab,True,False
https://doi.org/10.80240/FK2/DOAFHQ,Restricted 2,70219,odesi_datasetsno_borealis.png,False,False
https://doi.org/10.80240/FK2/PTJEYL,Restricted 1,70215,Demotestmetadata.json,False,True
CSV row count: 6


In [38]:
from pathlib import Path
text = Path('/usr/local/python/3.12.1/lib/python3.12/site-packages/pyDataverse/api.py').read_text().splitlines()
start = next(i for i,l in enumerate(text) if l.startswith('def update_datafile_metadata'))
end = next((i for i,l in enumerate(text[start+1:], start+1) if l.startswith('def ')), len(text))
print('start', start, 'end', end)
for i,line in enumerate(text[start:end], start):
    if i < start+120 or i > end-120:
        print(f'{i}: {line}')
    elif i == start+120:
        print('...')


StopIteration: 

In [21]:
import importlib.util
import pathlib
print('checking collection_to_csv syntax...')

spec = importlib.util.spec_from_file_location('collection_to_csv', pathlib.Path('collection_to_csv.py'))
module = importlib.util.module_from_spec(spec)
try:
    spec.loader.exec_module(module)
    print('collection_to_csv imported successfully')
except Exception as exc:
    print('import error:', exc)


checking collection_to_csv syntax...
collection_to_csv imported successfully


In [20]:
print(inspect.getsource(SearchApi.search)[1200:2400])

    url += "&fq={0}".format(filter_query)
        if show_entity_ids:
            url += "&show_entity_ids={0}".format(show_entity_ids)
        if query_entities:
            url += "&query_entities={0}".format(query_entities)
        return self.get_request(url, auth=auth)



In [27]:
import importlib.util
from pathlib import Path

spec = importlib.util.spec_from_file_location('push_json_to_dataverse', Path('push_json_to_dataverse.py'))
module = importlib.util.module_from_spec(spec)
try:
    spec.loader.exec_module(module)
    print('push_json_to_dataverse imported successfully')
except Exception as exc:
    print('import error:', exc)


push_json_to_dataverse imported successfully


In [40]:
import inspect
import pyDataverse
from pyDataverse.api import NativeApi
print('pyDataverse path:', pyDataverse.__file__)
print('NativeApi.update_datafile_metadata signature:', inspect.signature(NativeApi.update_datafile_metadata))
print('NativeApi methods:', [name for name in dir(NativeApi) if 'restrict' in name.lower() or 'update' in name.lower()])


pyDataverse path: /usr/local/python/3.12.1/lib/python3.12/site-packages/pyDataverse/__init__.py
NativeApi.update_datafile_metadata signature: (self, identifier, json_str=None, is_filepid=False)
NativeApi methods: ['restrict_datafile', 'update_datafile_metadata']


In [41]:
import inspect
from pyDataverse.api import NativeApi
print(inspect.getsource(NativeApi.update_datafile_metadata))
print('---')
print(inspect.getsource(NativeApi.restrict_datafile))


    def update_datafile_metadata(self, identifier, json_str=None, is_filepid=False):
        """Update datafile metadata.

        metadata such as description, directoryLabel (File Path) and tags are not carried over from the file being replaced:
        Updates the file metadata for an existing file where ID is the
        database id of the file to update or PERSISTENT_ID is the persistent id
        (DOI or Handle) of the file. Requires a jsonString expressing the new
        metadata. No metadata from the previous version of this file will be
        persisted, so if you want to update a specific field first get the
        json with the above command and alter the fields you want.


        Also note that dataFileTags are not versioned and changes to these will update the published version of the file.

        This functions needs CURL to work!

        HTTP Request:

        .. code-block:: bash

            POST -F 'file=@file.extension' -F 'jsonData={json}' http://$SERVER/api

In [42]:
from pyDataverse.api import NativeApi
print([name for name in dir(NativeApi) if name.endswith('_request')])
print('has post_request', hasattr(NativeApi, 'post_request'))


['_async_request', '_sync_request', 'delete_request', 'get_request', 'post_request', 'put_request']
has post_request True


In [43]:
import inspect
from pyDataverse.api import NativeApi
print(inspect.getsource(NativeApi.post_request))


    def post_request(
        self, url, data=None, auth=DEPRECATION_GUARD, params=None, files=None
    ):
        """Make a POST request.

        params will be added as key-value pairs to the URL.

        Parameters
        ----------
        url : str
            Full URL.
        data : str
            Metadata as a json-formatted string. Defaults to `None`.
        auth : Any
            .. deprecated:: 0.3.4
                The auth parameter was ignored before version 0.3.4.
                Please pass your auth to the Api instance directly, as
                explained in :py:func:`Api.__init__`.
                If you need multiple auth methods, create multiple
                API instances:

                .. code-block:: python

                    api = Api("https://demo.dataverse.org", auth=ApiTokenAuth("my_api_token"))
                    api_oauth = Api("https://demo.dataverse.org", auth=BearerTokenAuth("my_bearer_token"))
        files : dict
            e.g. :code:`

In [44]:
import inspect
from pyDataverse.api import NativeApi
print(inspect.getsource(NativeApi._check_json_data_form))


    @staticmethod
    def _check_json_data_form(data: Optional[Dict]):
        """This method checks and distributes given payload to match Dataverse expectations.

        In the case of the form-data keyed by "jsonData", Dataverse expects
        the payload as a string in a form of a dictionary. This is not possible
        using HTTPXs json parameter, so we need to handle this case separately.
        """

        if not data:
            return {}
        elif not isinstance(data, dict):
            raise ValueError("Data must be a dictionary.")
        elif "jsonData" not in data:
            return {"json": data}

        assert list(data.keys()) == [
            "jsonData"
        ], "jsonData must be the only key in the dictionary."

        # Content of JSON data should ideally be a string
        content = data["jsonData"]
        if not isinstance(content, str):
            data["jsonData"] = json.dumps(content)

        return {"data": data}



In [46]:
from pathlib import Path
import importlib.util

# Test the updated push_json_to_dataverse.py logic in dry-run mode.
import sys
sys.path.append('/workspaces/Bulk_RequestAccess_CollectionLevel')
from push_json_to_dataverse import push_dataset_json, parse_bool

native_api = None
print('Imported push_json_to_dataverse')
json_path = Path('/workspaces/Bulk_RequestAccess_CollectionLevel/dataset_jsons/dataset_10.80240_FK2_6RZD0P.json')
# Dry-run should count updates without requiring a server.
changed, errors = push_dataset_json(native_api, json_path, dry_run=True)
print('dry-run changed', changed, 'errors', errors)


Imported push_json_to_dataverse
DRY RUN: would update file 70236 in doi:10.80240/FK2/6RZD0P: restricted=True, fileAccessRequest=True
DRY RUN: would update file 70235 in doi:10.80240/FK2/6RZD0P: restricted=True, fileAccessRequest=True
DRY RUN: would update file 70234 in doi:10.80240/FK2/6RZD0P: restricted=True, fileAccessRequest=True
dry-run changed 3 errors 0


In [47]:
import inspect
from pyDataverse.api import NativeApi
api = NativeApi('https://demo.borealisdata.ca', api_token='X')
print('NativeApi init source:\n')
print(inspect.getsource(NativeApi.__init__))
print('base_url_api_native:', api.base_url_api_native)
print('base_url_api:', getattr(api, 'base_url_api', None))
print('base_url:', getattr(api, 'base_url', None))
print('base_url_api_native attr exists:', hasattr(api, 'base_url_api_native'))
print('all base_ attrs:', [a for a in dir(api) if a.startswith('base_')])


NativeApi init source:

    def __init__(self, base_url: str, api_token=None, api_version="v1", *, auth=None):
        """Init an Api() class.

        Scheme, host and path combined create the base-url for the api.
        See more about URL at `Wikipedia <https://en.wikipedia.org/wiki/URL>`_.

        Parameters
        ----------
        native_api_version : str
            API version of Dataverse native API. Default is `v1`.

        """
        super().__init__(base_url, api_token, api_version, auth=auth)
        self.base_url_api_native = self.base_url_api

base_url_api_native: https://demo.borealisdata.ca/api/v1
base_url_api: https://demo.borealisdata.ca/api/v1
base_url: https://demo.borealisdata.ca
base_url_api_native attr exists: True
all base_ attrs: ['base_url', 'base_url_api', 'base_url_api_native']


In [48]:
import importlib.util
from pathlib import Path
import inspect

spec = importlib.util.spec_from_file_location('push_json_to_dataverse', Path('push_json_to_dataverse.py'))
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
print('push_json_to_dataverse imported OK')
print('push_restrict source:')
print(inspect.getsource(module.push_restrict))
print('update_file_access_request source:')
print(inspect.getsource(module.update_file_access_request))


push_json_to_dataverse imported OK
push_restrict source:
def push_restrict(native_api, file_id, restricted, use_pid=False):
    if restricted:
        if use_pid:
            url = build_files_api_url(native_api, f"/files/:persistentId/restrict?persistentId={file_id}")
        else:
            url = build_files_api_url(native_api, f"/files/{file_id}/restrict")
    else:
        if use_pid:
            url = build_files_api_url(native_api, f"/files/:persistentId/unrestrict?persistentId={file_id}")
        else:
            url = build_files_api_url(native_api, f"/files/{file_id}/unrestrict")
    return native_api.put_request(url, auth=True)

update_file_access_request source:
def update_file_access_request(native_api, file_id, new_value, use_pid=False):
    if use_pid:
        url = build_files_api_url(native_api, f"/files/:persistentId/metadata?persistentId={file_id}")
    else:
        url = build_files_api_url(native_api, f"/files/{file_id}/metadata")

    data = {"jsonData": {"file